# 🔥 Wildfire Survival — End-to-End Walkthrough
**WiDS Global Datathon 2026 · Break Through Tech Spring AI Studio**

This notebook walks through our full pipeline for predicting the probability that a wildfire threatens an
evacuation zone within 12/24/48/72 hours, using only the first 5 hours of fire behavior.

**Flow:** load data → EDA → the distant-fire insight → feature engineering → survival ensemble →
per-horizon calibration → guardrails → submission.

> The reusable implementation lives in [`../src/`](../src/); this notebook is the narrative that ties it
> together. Add the Kaggle data to `../data/` first (see `../data/README.md`).

## 0. Setup

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))   # so `import src...` works from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config
from src.features import add_engineered_features, engineered_feature_columns
from src.ensemble import SurvivalEnsemble
from src.calibration import PerHorizonCalibrator, binary_target_by_horizon
from src.evaluate import evaluate_all, reliability_curve
from src.pipeline import prepare_features, enforce_monotonicity, zero_out_impossible_hits

pd.set_option("display.float_format", lambda v: f"{v:.4f}")
np.random.seed(config.RANDOM_STATE)
print("Horizons:", config.HORIZONS, "| hit radius (km):", config.HIT_RADIUS_KM)

## 1. Load the data
~316 wildfire events. Labels are right-censored: `event` = did it hit within 72h, `time_to_hit_hours` = when
(or the 72h censoring time).

In [ ]:
train = pd.read_csv(config.DATA_DIR / config.TRAIN_FILE)
test  = pd.read_csv(config.DATA_DIR / config.TEST_FILE)
print("train:", train.shape, "| test:", test.shape)
train.head()

## 2. Exploratory data analysis

In [ ]:
# Class balance: how many fires actually hit a zone within 72h?
print(train[config.EVENT_COL].value_counts(normalize=True).rename("proportion"))

# Distribution of observed hit times (for fires that hit).
hit = train[train[config.EVENT_COL] == 1]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist(hit[config.TIME_COL], bins=20, color="#d1495b")
ax[0].set(title="Time-to-hit (fires that hit)", xlabel="hours", ylabel="count")
for h in config.HORIZONS:
    ax[0].axvline(h, ls="--", c="gray", lw=0.8)
ax[1].hist(train["distance_to_zone_km"], bins=30, color="#3c6e71")
ax[1].axvline(config.HIT_RADIUS_KM, ls="--", c="red", label="5 km hit radius")
ax[1].set(title="Distance to nearest zone", xlabel="km"); ax[1].legend()
plt.tight_layout(); plt.show()

## 3. The key insight — distant-fire overconfidence
No fire in training ever hit a zone from beyond ~5 km. Any model assigning real hit probability to fires far
away is overconfident. Let's confirm the distance/outcome relationship that motivates our fixes.

In [ ]:
# Hit rate as a function of distance bucket — expect it to collapse to ~0 past a few km.
buckets = pd.cut(train["distance_to_zone_km"], bins=[0, 2, 5, 10, 25, 100, 1000])
rate = train.groupby(buckets)[config.EVENT_COL].agg(["mean", "count"])
rate.columns = ["hit_rate", "n_fires"]
print(rate)

# Farthest distance from which a fire actually hit:
max_hit_dist = train.loc[train[config.EVENT_COL] == 1, "distance_to_zone_km"].max()
print(f"\nFarthest distance a fire hit from: {max_hit_dist:.1f} km")

## 4. Feature engineering (+20 features)
Distance-decay, distance×speed interactions, directional, growth-dynamics, and composite risk scores.
See `src/features.py`.

In [ ]:
X_train, X_test, event, time, feature_cols = prepare_features(train, test)
print(f"{len(feature_cols)} model-input features")
print("Sample engineered features:",
      [c for c in feature_cols if c not in config.RAW_FEATURES][:12])

## 5. Cross-validated survival ensemble
XGBoost Cox + Random Survival Forest + Gradient Boosted Survival Trees, blended with per-horizon weights
learned on out-of-fold predictions. See `src/ensemble.py`.

In [ ]:
ensemble = SurvivalEnsemble().fit(X_train, event, time)
print("Per-horizon ensemble weights:")
ensemble.weight_table().round(2)

## 6. Per-horizon calibration
The biggest single improvement. We fit isotonic and Platt calibrators on out-of-fold predictions and keep the
better one (by Brier score) per horizon. See `src/calibration.py`.

In [ ]:
# Out-of-fold blended predictions on TRAIN, used to fit calibration honestly.
oof = ensemble._out_of_fold_predictions(X_train, event, time, config.HORIZONS)
oof_blend = np.zeros((len(X_train), len(config.HORIZONS)))
for j in range(len(config.HORIZONS)):
    stack = np.stack([oof[n][:, j] for n in ensemble.model_names], axis=1)
    oof_blend[:, j] = stack @ ensemble.weights_[:, j]

calibrator = PerHorizonCalibrator().fit(oof_blend, event, time)
print("Calibrator chosen per horizon:", calibrator.summary())
print("\nPre-calibration :", evaluate_all(oof_blend, event, time))
print("Post-calibration:", evaluate_all(calibrator.transform(oof_blend), event, time))

### Reliability diagrams — before vs. after calibration

In [ ]:
cal_blend = calibrator.transform(oof_blend)
fig, axes = plt.subplots(1, len(config.HORIZONS), figsize=(14, 3.2), sharex=True, sharey=True)
for j, (h, ax) in enumerate(zip(config.HORIZONS, axes)):
    y = binary_target_by_horizon(event, time, h)
    for probs, label, c in [(oof_blend[:, j], "raw", "#d1495b"),
                            (cal_blend[:, j], "calibrated", "#3c6e71")]:
        mp, obs = reliability_curve(probs, y)
        ax.plot(mp, obs, "o-", color=c, label=label, ms=4)
    ax.plot([0, 1], [0, 1], "--", c="gray", lw=0.8)
    ax.set(title=f"{h}h", xlabel="predicted"); ax.set_ylim(-0.02, 1.02)
axes[0].set_ylabel("observed"); axes[0].legend()
plt.tight_layout(); plt.show()

## 7. Predict, apply guardrails, and build the submission
Calibrate the test predictions, clamp physically-impossible hits to 0, and enforce monotonicity across
horizons.

In [ ]:
test_probs = ensemble.predict_hit_prob(X_test, config.HORIZONS)
test_probs = calibrator.transform(test_probs)
test_probs = zero_out_impossible_hits(test, test_probs)
test_probs = enforce_monotonicity(test_probs)

submission = pd.DataFrame({config.ID_COL: test[config.ID_COL].to_numpy()})
for j, h in enumerate(config.HORIZONS):
    submission[f"prob_{h}h"] = test_probs[:, j]

out = config.SUBMISSIONS_DIR / "submission.csv"
submission.to_csv(out, index=False)
print("wrote", len(submission), "rows to", out)
submission.head()

## 8. Recap
- Survival framing keeps the four horizons consistent and uses the censored labels properly.
- The decisive problem was **calibration**, not ranking — distant fires were overconfident.
- Feature engineering + a CV-weighted ensemble + **per-horizon calibration** + physical guardrails took the
  baseline from **0.87397 → 0.96366**, clearing the 0.90 target.

See [`../docs/methodology.md`](../docs/methodology.md) for the full reasoning and
[`../docs/results.md`](../docs/results.md) for the ablation.